# 00. Data Collection - About this notebook

This notebook downloads and saves all raw data needed for the Energy Procurement Optimization project.

**Three data sources:**
| # | Source | What it provides | How accessed | Data Period  |
|---|--------|------------------|--------------|--------------|
| 1 | DAEWOO Steel UCI/Kaggle | Hourly demand profile D(t) | CSV download | 2018 |
| 2 | ENTSO-E Transparency Platform | Day-ahead price p_3(t) of Finland (FI) | CSV download | 2018 |
| 3 | Fingrid Open Data | Balancing price p_bal(t) & emission factor ci(t)| REST API | 2018 |

**Date alignment strategy:**  
We treat them as independent proxies: consumption *patterns* from DAEWOO, and electricity market *data* from 2018 Finland markets.

In [1]:
#SET UP 
import os
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from entsoe import EntsoePandasClient

ROOT_DIR = Path(__file__).parent.parent if '__file__' in dir() else Path().resolve().parent
RAW_DIR = ROOT_DIR / 'data' / 'raw'
PROCESSED_DIR = ROOT_DIR / 'data' / 'processed'

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 1. DAEWOO Steel Consumption Data
**Dataset:** Steel Industry Energy Consumption   
**Source:** UCI Machine Learning Repository 
**URL:** https://archive.ics.uci.edu/dataset/851/steel+industry+energy+consumption 

**Key columns we're gonna use:**
- `date` — timestamp (15-min intervals) → we will merge into hourly data → time context (hour) scenario labels
- `Usage_kWh` - electricity consumption → our demand D(t)
- `Load_Type` - Light Load / Medium Load / Maximum Load → demand scenario labels

In [7]:
#Load data & Quick inspect
daewoo_raw = pd.read_csv(RAW_DIR / "daewoo_steel.csv")
print('Shape:', daewoo_raw.shape)
print('\nColumns:', daewoo_raw.columns)
print('\nFirst rows:')
daewoo_raw.head(3)

Shape: (35040, 11)

Columns: Index(['date', 'Usage_kWh', 'Lagging_Current_Reactive.Power_kVarh',
       'Leading_Current_Reactive_Power_kVarh', 'CO2(tCO2)',
       'Lagging_Current_Power_Factor', 'Leading_Current_Power_Factor', 'NSM',
       'WeekStatus', 'Day_of_week', 'Load_Type'],
      dtype='object')

First rows:


,date,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,WeekStatus,Day_of_week,Load_Type
0,01/01/2018 00:15,3.17,2.95,0.0,0.0,73.21,100.0,900,Weekday,Monday,Light_Load
1,01/01/2018 00:30,4.00,4.46,0.0,0.0,66.77,100.0,1800,Weekday,Monday,Light_Load
2,01/01/2018 00:45,3.24,3.28,0.0,0.0,70.28,100.0,2700,Weekday,Monday,Light_Load


In [8]:
# Resample data into hourly 
daewoo_raw['date'] = pd.to_datetime(daewoo_raw['date'],dayfirst=True) #for consistent datetime format
daewoo_raw = daewoo_raw.loc[:,['date','Usage_kWh','Load_Type']].set_index('date').sort_index()

# Currently is energy consumed per 15-min interval, so that we take sum by hour
daewoo_hourly = daewoo_raw[['Usage_kWh']].resample('h').sum()
# Load type: we determine by the most common load type in 4 15-min intervals in each hour 
hourly_loadtype = daewoo_raw[['Load_Type']].resample('h').agg(
    lambda x: x.mode() if len(x) > 0 else np.nan
)
# We add other features (Time context: Weekend/Weekday, Hour: Day/Night/Evening)
daewoo_hourly = daewoo_hourly.join(hourly_loadtype)
daewoo_hourly['hour']       = daewoo_hourly.index.hour
daewoo_hourly['weekday']    = daewoo_hourly.index.dayofweek  
daewoo_hourly['is_weekday'] = (daewoo_hourly['weekday'] < 5).astype(int)
daewoo_hourly['time_of_day'] = pd.cut(
    daewoo_hourly['hour'],
    bins=[-1, 6, 17, 23],
    labels=['Night', 'Day', 'Evening']
)

print(f'Hourly records: {len(daewoo_hourly)}')
print(f'Date range: {daewoo_hourly.index.min()} → {daewoo_hourly.index.max()}')
print(f'\nLoad_Type distribution:')
print(daewoo_hourly['Load_Type'].value_counts())
daewoo_hourly.head(3)

Hourly records: 8760
Date range: 2018-01-01 00:00:00 → 2018-12-31 23:00:00

Load_Type distribution:
Load_Type
Light_Load      4518
Medium_Load     2424
Maximum_Load    1818
Name: count, dtype: int64


,Usage_kWh,Load_Type,hour,weekday,is_weekday,time_of_day
date,,,,,,
2018-01-01 00:00:00,13.83,Light_Load,0,0,1,Night
2018-01-01 01:00:00,14.01,Light_Load,1,0,1,Night
2018-01-01 02:00:00,14.12,Light_Load,2,0,1,Night


In [6]:
# DAEWOO annual consumption 
print(f'Annual electricity consumption of DAEWOO is: {daewoo_hourly['Usage_kWh'].sum()/1e3:.2f} MWh')
print(f'-> Thus the result obtained is for a representative firm with annual consumption of {daewoo_hourly['Usage_kWh'].sum()/1e6:.1f} GWh')

Annual electricity consumption of DAEWOO is: 959.64 MWh
-> Thus the result obtained is for a representative firm with annual consumption of 1.0 GWh


In [10]:
# Scale up to match Finnish Industry 
# Finnish steel industry average consumption >= 500 GWh/year (eg. SSAB)

# Target: 500 GWh/year = 500,000 MWh/year
target_gwh = 500
scale_factor = (target_gwh * 1e6) / daewoo_hourly['Usage_kWh'].sum()  # kWh scaled
print(f'Scale factor: {scale_factor:.2f}x')

daewoo_hourly['demand_mwh']  = daewoo_hourly['Usage_kWh'] * scale_factor/ 1000

print(f'Updated annual consumption: {daewoo_hourly['demand_mwh'].sum()/1000:.2f} GWh')

Scale factor: 521.03x
Updated annual consumption: 500.00 GWh


In [14]:
# Save processed data
daewoo_hourly = daewoo_hourly.iloc[:,-6:]
daewoo_hourly.to_csv(PROCESSED_DIR/ 'daewoo_hourly.csv')

# 2. ENTSO-E: Finnish Day-ahead Spot Prices (2018)
Finland Day-Ahead Spot Prices -> $p_3(t)$ in the model

**Source:** ENTSO-E Transparency <br>
**URL:** https://transparency.entsoe.eu <br>
**Bidding zone:** Finland <br>
**Period:** Full year 2018 (to align with Fingrid data)  

In [15]:
# Laod data and quick inspection
spot_price = pd.read_csv(RAW_DIR/ 'spot_price.csv')
spot_price.head(3)

,MTU (CET/CEST),Area,Sequence,Day-ahead Price (EUR/MWh),Intraday Period (CET/CEST),Intraday Price (EUR/MWh)
0,01/01/2018 00:00:00 - 01/01/2018 01:00:00,BZN|FI,Without Sequence,26.33,NaN,NaN
1,01/01/2018 01:00:00 - 01/01/2018 02:00:00,BZN|FI,Without Sequence,26.43,NaN,NaN
2,01/01/2018 02:00:00 - 01/01/2018 03:00:00,BZN|FI,Without Sequence,26.10,NaN,NaN


In [16]:
# Format date time to match with other dataset
spot_price = spot_price.loc[:,['MTU (CET/CEST)','Day-ahead Price (EUR/MWh)']]
def parse_datetime(s):
    formats = [
        '%d/%m/%Y %H:%M:%S',
        '%Y-%m-%d %H:%M:%S',
        '%d-%m-%Y %H:%M:%S',
        '%Y/%m/%d %H:%M:%S',
    ]
    for fmt in formats:
        try:
            return pd.Timestamp(s.strip(), )
        except:
            pass

spot_price['date'] = spot_price['MTU (CET/CEST)'].str.split(' - ').str[0].apply(parse_datetime)

In [17]:
# We check timestamp at which timezone changes: 
spot_price['tz_flag'] = spot_price['MTU (CET/CEST)'].str.extract(r'\((CET|CEST)\)')
spot_price[spot_price['tz_flag'].notna()]

,MTU (CET/CEST),Day-ahead Price (EUR/MWh),date,tz_flag
1992,25/03/2018 00:00:00 - 25/03/2018 01:00:00 (CET),38.52,2018-03-25 00:00:00,CET
1993,25/03/2018 01:00:00 (CET) - 25/03/2018 03:00:0...,38.01,NaT,CET
1994,25/03/2018 03:00:00 (CEST) - 25/03/2018 04:00:00,37.85,NaT,CEST
7200,28/10/2018 01:00:00 - 28/10/2018 02:00:00 (CEST),43.58,2018-10-28 01:00:00,CEST
7201,28/10/2018 02:00:00 (CEST) - 28/10/2018 02:00:...,41.62,NaT,CEST
7202,28/10/2018 02:00:00 (CET) - 28/10/2018 03:00:00,41.59,NaT,CET


In [18]:
# Work manually on a copy
spot_clean = spot_price.copy()

# 25 March 2018
# Row 1993 covers 01:00–03:00 (2 hours) -> we assign to 01:00
spot_clean.loc[1993, 'date'] = pd.Timestamp('2018-03-25 01:00:00')
# For missing 02:00, we assign same price as 1993 (same interval)
row_02 = spot_clean.loc[1993].copy()
row_02['date'] = pd.Timestamp('2018-03-25 02:00:00')
# Row 1994, 03:00 CEST - keep as is
spot_clean.loc[1994, 'date'] = pd.Timestamp('2018-03-25 03:00:00')
spot_clean = pd.concat([spot_clean, pd.DataFrame([row_02])])

# 28 Oct 2018
# Row 7200 01:00 CEST keep asis
# For 02:00, there are 2 occurences before and after timezone changes, thus we take the average value of 
# Row 7201 & 7202 (drop these two rows & add avg. value row)
avg_price = (spot_clean.loc[7201, 'Day-ahead Price (EUR/MWh)'] + 
             spot_clean.loc[7202, 'Day-ahead Price (EUR/MWh)']) / 2
spot_clean = spot_clean.drop(index=[7201, 7202])
row_avg = spot_clean.loc[7200].copy()
row_avg['date'] = pd.Timestamp('2018-10-28 02:00:00')
row_avg['Day-ahead Price (EUR/MWh)'] = avg_price
spot_clean = pd.concat([spot_clean, pd.DataFrame([row_avg])])

#Final cleaned file
spot_clean = spot_clean.sort_values('date')

In [19]:
print(f'Number of observations: {spot_clean.shape[0]}')
print(f'Duplicated timestamps: {spot_clean.duplicated('date', keep=False).sum()}')
spot_clean.head(3)

Number of observations: 8760
Duplicated timestamps: 0


,MTU (CET/CEST),Day-ahead Price (EUR/MWh),date,tz_flag
0,01/01/2018 00:00:00 - 01/01/2018 01:00:00,26.33,2018-01-01 00:00:00,NaN
1,01/01/2018 01:00:00 - 01/01/2018 02:00:00,26.43,2018-01-01 01:00:00,NaN
2,01/01/2018 02:00:00 - 01/01/2018 03:00:00,26.10,2018-01-01 02:00:00,NaN


In [20]:
# Save processed data
spot_clean = spot_clean.loc[:,['Day-ahead Price (EUR/MWh)','date']].set_index('date').sort_index()
spot_clean.to_csv(PROCESSED_DIR/ 'day_ahead_price.csv')

# 3. Fingrid: Balancing Prices & Emission factor
**Source:** Fingrid Open Data API  (The personal API key is retracted here. To download data, replace with your API key)<br>
**URL:** https://data.fingrid.fi  <br>

**a) Balancing price** for **p_bal(t)** in the model <br>
Dataset ID: 244 <br>

**b) Emission factor** for electricity consumed **ci(t)** in the model <br>
Dataset ID: 265 <br>

In [23]:
def fetch_fingrid(variable_id: int, start: str, end: str) -> pd.DataFrame:
    """
    Fetch data from the Fingrid API.

    Parameters:
    #variable_id: Dataset ID
    #start, end: Data Period to extract
    
    """
    base_url    = f"https://data.fingrid.fi/api/datasets/{variable_id}/data"
    headers     = {"x-api-key": FINGRID_API_KEY}
    page        = 1
    page_size   = 10000
    all_records = []
 
    while True:
        params = {
            "startTime":  START,
            "endTime":    END,
            "format":     "json",
            "pageSize":   page_size,
            "page":       page,
            "sortBy":     "startTime",
            "sortOrder":  "asc",
        }
 
        resp = requests.get(base_url, headers=headers, params=params, timeout=30)

        resp.raise_for_status()
        data = resp.json()
 
        records = data.get("data", [])
        if not records:
            break
 
        all_records.extend(records)
 
        pagination  = data.get("pagination", {})
        total_pages = pagination.get("lastPage", 1)
        if page >= total_pages:
            break

    
    df = pd.DataFrame(all_records)
    return df
START = '2018-01-01T00:00:00Z'
END   = '2019-01-01T00:00:00Z'
#FINGRID_API_KEY = 'xyz123' #Replace with your API key from Fingrid
# Fetch balancing_price data
#balancing_price = fetch_fingrid(244, START, END)
#balancing_price.to_csv(RAW_DIR/ 'bal_price.csv')
#emission_factor = fetch_fingrid(265, START, END)
#emission_factor.to_csv(RAW_DIR/ 'emission_factor.csv')

## 3.1. Balancing Price

In [325]:
# Load and process data
pbal = pd.read_csv(RAW_DIR/'bal_price.csv')
print(f'Number of observation: {pbal.shape[0]}')
print(f'Missing values: {pbal.isna().sum()}')
pbal.head(3)

Number of observation: 8760
Missing values: Unnamed: 0    0
datasetId     0
startTime     0
endTime       0
value         0
dtype: int64


,Unnamed: 0,datasetId,startTime,endTime,value
0,0,244,2018-01-01T00:00:00.000Z,2018-01-01T01:00:00.000Z,30.46999
1,1,244,2018-01-01T01:00:00.000Z,2018-01-01T02:00:00.000Z,30.00000
2,2,244,2018-01-01T02:00:00.000Z,2018-01-01T03:00:00.000Z,30.00000


In [326]:
# Format date: 
pbal['date'] = pd.to_datetime(pbal['startTime']).dt.tz_localize(None)
pbal = pbal.loc[:,['date','value']].set_index('date').sort_index()
pbal = pbal.rename(columns={'value':'balancing_price'})
pbal.head(3)

,balancing_price
date,
2018-01-01 00:00:00,30.46999
2018-01-01 01:00:00,30.00000
2018-01-01 02:00:00,30.00000


In [327]:
# Save processed data
pbal.to_csv(PROCESSED_DIR/ 'balancing_price.csv')

## 3.2. Emission Factor

In [83]:
# Load and process data
emission = pd.read_csv(RAW_DIR/'emission_factor.csv',sep=';').sort_values('startTime')
emission.head(3)

,startTime,endTime,Emission factor for electricity consumed in Finland - real time data
0,2017-12-31T22:03:00.000Z,2017-12-31T22:03:00.000Z,132.16000
1,2017-12-31T22:06:00.000Z,2017-12-31T22:06:00.000Z,130.47999
2,2017-12-31T22:09:00.000Z,2017-12-31T22:09:00.000Z,127.69000


In [84]:
# Group by hour
emission = emission.rename(columns={'Emission factor for electricity consumed in Finland - real time data':'emission'})
emission['date'] = pd.to_datetime(emission['startTime']).dt.tz_localize(None)
emission = emission[(emission['date'] >= '2018-01-01') & (emission['date'] < '2019-01-01')]
emission = emission.set_index('date').sort_index()
emission = emission[['emission']].resample('h').mean()

# Fingrid dat unit is gCO2/kwh -> we transform for mwh unit: 
emission['ci_gco2_mwh'] = emission['emission'] * 1e3
emission.head(3)

,emission,ci_gco2_mwh
date,,
2018-01-01 00:00:00,107.207999,107207.9990
2018-01-01 01:00:00,105.856999,105856.9990
2018-01-01 02:00:00,117.172999,117172.9995


In [85]:
#Save data
emission = emission.loc[:,'ci_gco2_mwh']
emission.to_csv(PROCESSED_DIR / 'emission_factor.csv')

# 4. AGGREGATE DATASET

In [89]:
spot_price = pd.read_csv(PROCESSED_DIR/'day_ahead_price.csv')
balancing_price = pd.read_csv(PROCESSED_DIR/'balancing_price.csv')
daewoo_hourly = pd.read_csv(PROCESSED_DIR/'daewoo_hourly.csv')
emission = pd.read_csv(PROCESSED_DIR/'emission_factor.csv')

In [90]:
dataset = daewoo_hourly.merge(spot_price, on = 'date',how='left').merge(
    balancing_price, on = 'date',how='left').merge(
    emission, on ='date', how='left'
    )
print(f'Missing values: {dataset.isna().sum()}')
dataset.head(3)

Missing values: date                         0
Load_Type                    0
hour                         0
weekday                      0
is_weekday                   0
time_of_day                  0
demand_mwh                   0
Day-ahead Price (EUR/MWh)    0
balancing_price              0
ci_gco2_mwh                  1
dtype: int64


,date,Load_Type,hour,weekday,is_weekday,time_of_day,demand_mwh,Day-ahead Price (EUR/MWh),balancing_price,ci_gco2_mwh
0,2018-01-01 00:00:00,Light_Load,0,0,1,Night,7.205852,26.33,30.46999,107207.9990
1,2018-01-01 01:00:00,Light_Load,1,0,1,Night,7.299637,26.43,30.00000,105856.9990
2,2018-01-01 02:00:00,Light_Load,2,0,1,Night,7.356951,26.10,30.00000,117172.9995


In [91]:
dataset = dataset.rename(columns={'Usage_kWh':'demand_kwh',
                                 'Load_Type':'load_type',
                                 'Day-ahead Price (EUR/MWh)':'dayahead_price'})
dataset.to_csv(PROCESSED_DIR/'dataset.csv')

---
**Next notebook:** `01_eda.ipynb` — Exploratory Data Analysis